In [33]:
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import numpy as np
import re
import cartopy.crs as ccrs
import cartopy

In [2]:
fpath = '/badc/deposited2018/robust-storm-track/data/era-20c/'
filename = fpath + 'ERA20C_tr_trs_VOR850_19802009_pos.addmslp_addspeed_addprecip_addomega_addlandwinds_addavgprecip.new_1000km2dayfiltered_RealProjregion2filtered_maxlandwindsinregion.txt'

with open(filename, 'r') as file:
    for i in range(10):
        print(file.readline().strip())

In [10]:
def read_tracks(filename=None, var_names=None):
    with open(filename, 'r') as file:
        lines = file.readlines()

    # Checking file existence and reading lines
    assert len(lines) > 0, "File does not exist or is empty"

    # Find all lines starting with "TRACK_ID"
    i_start = [i for i, line in enumerate(lines) if line.startswith('TRACK_ID')]
    n_tracks = len(i_start)

    tracks = []
    for ii in range(n_tracks):
        start_ = i_start[ii] + 2
        n_points = int(lines[i_start[ii] + 1].split()[1])
        track_lines = lines[start_:start_ + n_points]

        # Cleaning and converting data
        track_lines = [re.sub(r'&', '', line).split() for line in track_lines]
        data = np.array(track_lines, dtype=float)
        data[data == 1e25] = np.nan  # Setting NA values

        # Create DataFrame from data
        df = pd.DataFrame(data)
        track_id = '_'.join(lines[i_start[ii]].split()[2:4])

        # Assigning the DataFrame an ID based on the track
        df['ID'] = track_id
        tracks.append(df)

    # Combining all tracks into a single DataFrame
    result = pd.concat(tracks, ignore_index=True)

    # Renaming columns based on var_names if provided
    if var_names:
        for idx, name in enumerate(var_names, start=0):
            result.rename(columns={idx: name}, inplace=True)

    return result

def add_wrap_id(df):
    # Convert all column names to strings to prevent AttributeError
    df.columns = [str(col) for col in df.columns]

    # Identify all longitude columns
    lon_cols = [col for col in df.columns if col.startswith('lon_')]

    for lon_col in lon_cols:
        suffix = lon_col.split('_')[-1]  # Extract the suffix after 'lon_'
        lat_col = f'lat_{suffix}'  # Assume the corresponding latitude column

        # Define a function to calculate wrap changes
        def calculate_wrap_id(lon):
            # Detect crossings of the dateline
            diffs = np.diff(np.concatenate(([lon.iloc[0]], lon)))  # Include the first element for continuity
            wraps = diffs > 180  # Change ID when crossing the dateline
            return np.cumsum(wraps) + 1  # Start IDs at 1 and increment on wrap

        # Apply the function across the dataframe grouped by 'ID' if it exists
        if 'ID' in df.columns:
            df[f'wrap_{suffix}'] = df.groupby('ID')[lon_col].transform(calculate_wrap_id)
            df[f'ID2_{suffix}'] = df['ID'].astype(str) + '.' + df[f'wrap_{suffix}'].astype(str)
        else:
            df[f'wrap_{suffix}'] = calculate_wrap_id(df[lon_col])
            df[f'ID2_{suffix}'] = df.index.astype(str) + '.' + df[f'wrap_{suffix}'].astype(str)

        df.drop(f'wrap_{suffix}', axis=1, inplace=True)  # Remove temporary wrap column if not needed
    
    # Add 'TRACK_ID' column filled with the track ID
    df['TRACK_ID'] = df['ID'].str.split('_').str[-1]

    return df

In [13]:
var_names = ['date', 'lon_vor', 'lat_vor', 'var_vor', 'lon_mslp', 'lat_mslp', 'var_mslp',
            'lon_wind', 'lat_wind', 'var_wind', 'lon_prec', 'lat_prec', 'var_prec',
            'lon_ver', 'lat_ver', 'var_ver', 'lon_wind_sca', 'lat_wind_sca', 'var_wind_sca',
            'prec5deg', 'lon_cmorph_prec', 'lat_cmorph_prec', 'var_prec_max', 'var_prec_ave']

# Read the track data
trx = read_tracks(filename, var_names)
trx = add_wrap_id(trx)
trx

           date     lon_vor    lat_vor   var_vor  lon_mslp  lat_mslp  \
0  1.980100e+09  276.477875  32.726715  3.683265  278.2872  30.83150   
1  1.980100e+09  276.621796  32.233791  3.502314  276.9048  30.58227   
2  1.980100e+09  276.667175  31.761646  3.099555       NaN       NaN   
3  1.980100e+09  276.765594  31.485422  3.484782       NaN       NaN   
4  1.980100e+09  277.338654  31.542223  3.316391       NaN       NaN   

   var_mslp  lon_wind  lat_wind  var_wind  ...  var_wind_sca  prec5deg  \
0  1006.408  281.2500   35.4385  16.05729  ...      16.05729  0.458976   
1  1005.997  281.9531   36.1403  16.49195  ...      16.49195  0.429037   
2  1007.540  281.2500   35.4385  17.06239  ...      17.06239  0.430904   
3  1007.055  280.5469   34.7368  15.26498  ...      15.26498  0.317605   
4  1008.109  284.0625   30.5263  13.23500  ...      13.20241  0.325036   

                      ID                  ID2_vor                 ID2_mslp  \
0  START_TIME_1980100100  START_TIME_1980100

In [ ]:
trx['lon_vor'] = trx['lon_vor'].apply(lambda lon: ((lon + 180) % 360) - 180)

# Set up the map projection and the figure
fig, ax = plt.subplots(figsize=(10, 8.),
                       subplot_kw={'projection': ccrs.PlateCarree(central_longitude=0)})

# Add features to the map
ax.add_feature(cartopy.feature.COASTLINE, zorder=17)
# Color map
cmap = plt.cm.viridis

# Group data by 'date' and plot each group
for name, group in trx.groupby('TRACK_ID'):
    points = group[['lon_vor', 'lat_vor']].values
    lc = ccrs.Geodetic()
    ax.scatter(group['lon_vor'], group['lat_vor'], color=cmap(norm(group['var_vor'])), s=1, transform=lc)  # scatter for color variation
    for i in range(len(points)-1):
        line = np.array([points[i], points[i+1]])
        ax.plot(line[:, 0], line[:, 1], color=cmap(norm((group['var_vor'].iloc[i] + group['var_vor'].iloc[i+1])/2)),
                linewidth=0.5, transform=lc)  


# Group data by 'date' and plot each group
#for name, group in trx.groupby('TRACK_ID'):
#    ax.plot(group['lon_vor'], group['lat_vor'], linewidth=0.3)

gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
gl.top_labels = False
gl.right_labels = False

# Add a title and legend to the plot
ax.set_title('Extratropical Cyclone in Europe (1980-2010)')
plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, orientation='horizontal', label='Relative Vorticity (x10-5 s-1)')

# Display the plot
plt.show()

In [11]:
f = open(file,'r')
print(f)
rd = f.readline()
print(rd)
rd = f.readline()
rd = f.readline().split()
print(rd)
TRACK_NUM = int(rd[1])

for n in range(TRACK_NUM):
    rd = f.readline().split()
    print(rd)
    TRACK_ID = int(rd[1])
    print(TRACK_ID)
    rd = f.readline().split()
    POINT_NUM = int(rd[1])
    TRACK_TIME=[]; TRACK_DATE=[];TRACK_LON=[]; TRACK_LAT=[]; TRACK_VOR=[];TRACK_WND10=[];TRACK_MSLP=[];TRACK_MONTH=[];TRACK_YEAR=[]
    TRACK_WIND=[]; radius_10m=[]; radius_925=[]
    for i in range(POINT_NUM):
       rd = f.readline().split()
       TRACK_TIME.append(int(rd[0].lstrip('0')))
       TRACK_DATE.append(int(rd[0][4:8].lstrip('0')))
       TRACK_MONTH.append(int(rd[0][4:6].lstrip('0')))
       TRACK_YEAR.append(int(rd[0][:4].lstrip('0')))
       TRACK_LON.append(float(rd[1].lstrip('0')))
       TRACK_LAT.append(float(rd[2].lstrip('0')))
       TRACK_VOR.append(float(rd[3].lstrip('0')))
       TRACK_WIND.append(float(rd[-6].lstrip('0')))
       radius_10m.append(float(rd[-2].lstrip('0')))
       radius_925.append(float(rd[-4].lstrip('0')))

<_io.TextIOWrapper name='/badc/deposited2018/robust-storm-track/data/era-20c/ERA20C_tr_trs_VOR850_19802009_pos.addmslp_addspeed_addprecip_addomega_addlandwinds_addavgprecip.new_1000km2dayfiltered_RealProjregion2filtered_maxlandwindsinregion.txt' mode='r' encoding='UTF-8'>
0

['0', '0']
